# LIBRERIAS Y CARGA DEL MODELO

# QWEN V2.5

In [ ]:
# Isntalación de dependencias necesarias para ejecutar el proyecto
!pip install fastapi uvicorn python-multipar -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflaredt
!pip install bitsandbytes -q
!pip install "transformers>=4.49.0" -q
!pip install "qwen-vl-utils>=0.0.10" -q
print("✓ Listo. Reinicia el kernel y ejecuta desde la siguiente celda.")

In [1]:
# Librerias para el servidor FastAPI y el modelo Qwen2.5-VL.
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, HTMLResponse
import uvicorn, threading, io, subprocess, time, re
from PIL import Image
import json
from pathlib import Path

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

import cv2
import numpy as np
import json

import os
import glob

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

In [3]:
DEVICE

'cuda'

In [ ]:
# Solo aplicar si vas a cuantizar
# Para modelo de 4 bits (Cuantizacion)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [4]:
# Cargar del modelo y procesador
qwen_processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256  * 28 * 28,
    max_pixels= 2500*28*28 #1280 * 28 * 28,
)

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# Carga del modelo sin cuantizar
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    #quantization_config=bnb_config,   #← comentado = cuantización desactivada
    device_map="auto",
)
qwen_model.eval()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): SiLUAc

In [6]:
# Info del vram utilizado
if torch.cuda.is_available():
    usado = torch.cuda.memory_allocated() / 1e9
    libre = torch.cuda.get_device_properties(0).total_memory / 1e9 - usado
    print(f"✓ Qwen2.5-VL-7B cargado | VRAM usada: {usado:.1f} GB | libre: {libre:.1f} GB")

✓ Qwen2.5-VL-7B cargado | VRAM usada: 7.6 GB | libre: 8.1 GB


In [7]:
# Filtros usados

def upscale_lanczos_cv2(img, factor=2):
    h, w = img.shape[:2]
    return cv2.resize(img, (w * factor, h * factor), interpolation=cv2.INTER_LANCZOS4)

def _clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    return cv2.cvtColor(cv2.merge([clahe.apply(L), A, B]), cv2.COLOR_LAB2BGR)

def _sharp(img):
    k = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
    return cv2.addWeighted(img, 0.4, cv2.filter2D(img, -1, k), 0.6, 0)

# Funcion que hace el procesamiento de imagenes
def preprocesar2(img):
    
    img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    img = upscale_lanczos_cv2(img)
    img = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    img = _clahe(img)
    img = _sharp(img)

    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))


MAX_PIXELS = 2500 * 28 * 28   # el mismo que pusiste en el processor

def preprocesar(img, filtros_baja_res=True):
    w, h = img.size
    pixeles = w * h

    # Imagen de buena calidad: no tocar
    if pixeles >= MAX_PIXELS:
        return img

    # Imagen de baja resolución: aplicar filtros
    if filtros_baja_res:
        arr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        # Escalar solo lo necesario para llegar a MAX_PIXELS (no un ×2 fijo)
        factor = (MAX_PIXELS / pixeles) ** 0.5
        arr = cv2.resize(arr, (int(w * factor), int(h * factor)),
                         interpolation=cv2.INTER_LANCZOS4)
        arr = _clahe(arr)
        arr = _sharp(arr)
        return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))

    return img

In [ ]:
PROMPT_BASE = """
Eres un sistema experto en transcripción de documentos históricos y registrales del Perú.
Tu tarea es TRANSCRIBIR TODO el texto visible en la imagen, de forma literal y ordenada.
NO extraigas campos, NO resumas, NO interpretes. Solo transcribe y organiza.

Analiza la imagen y determina primero a cuál de los dos formatos corresponde:
- TIPO A: Manuscrito Antiguo (2 columnas: lateral izquierdo + declaración central)
- TIPO B: Tabular Moderno RENIEC (bloques con casillas)

Aplica ÚNICAMENTE las reglas del tipo que corresponda. Las reglas de un tipo no aplican al otro.
Debes analizar la imagen adjunta y extraer la información solicitada.

INSTRUCCIONES SEGÚN EL TIPO DE ACTA:
"""

PROMPT_F1 = """
1. SI EL ACTA TIENE 2 columnas - Declaracion lateral izquierdo / Declaración central derecha:
   A. INFORMACIÓN DE LA SECCION IZQUIERDA:
    Extraer toda información de la imagen. 

   B. SECCION DERECHA
	En esta sección deberas extraer toda la información tal cual se presenta. 
    También encontrar un código de barras. Deberas poner esa información en una 
    sección llamada "Datos barras" y colocar los números que figuran.
    """

PROMPT_F2 = """
SI EL ACTA ES TABULAR RENIEC ES DECIR la imagen es un acta con formato de tabla y bloques horizontales. 
Las etiquetas de cada campo están **DEBAJO** de las casillas punteadas, no encima.

**REGLA DE ORO (AISLAMIENTO TOTAL):**
Cada bloque es independiente. **PROHIBIDO** usar información de un bloque para rellenar otro. 
Si un campo no está en su bloque, déjalo como `null`.

---

### BLOQUE 1: OFICINA REGISTRAL (Parte superior izquierda)
- **Fecha de registro**: día/mes/año (ej. 21/12/2000). Está en la primera fila.
- **Distrito**: (ej. JESÚS MARÍA). 
- **Provincia**: (ej. LIMA).
- **Departamento**: (ej. LIMA).
- **Oficina registral**: Distrito, Provincia, Departamento (ej. "Jesús María, Lima, Lima").

### BLOQUE 2: DATOS DEL NACIDO
- **Apellidos**: En la fila "Apellidos" (ej. CÓRDOVA AGURTO).
- **Nombres**: En la fila "Nombres" (ej. ANDREA SUSANA).
- **Sexo**: En la fila "Sexo" (ej. 2 FEMENINO).
- **Fecha de nacimiento**: En las celdas de día/mes/año (ej. 21/12/2000).
- **Hora de nacimiento**: En la celda "Hora" (ej. 11:25 AM).
- **Lugar de nacimiento**: En las celdas "Departamento", "Provincia", "Distrito" (ej. "Hospital Rebagliati, Lima, Lima").

**Nombre completo del nacido**: Concatena `[Nombres] + [Apellidos]` (ej. "Andrea Susana Córdova Agurto").

### BLOQUE 3: DATOS DE LA MADRE
- **Apellidos**: En la fila "Apellidos" (ej. AGURTO MORAN).
- **Nombres**: En la fila "Nombres" (ej. SUSANA ISABEL).
- **DNI**: En la columna "DNI" (ej. 03690243).
- **Edad**: En la columna "Edad" (ej. 22).
- **Lugar de nacimiento**: En celdas "Departamento, Provincia, Distrito" (ej. "Piura, Piura, Piura").
- **Domicilio**: En la celda "Domicilio" (ej. "Jr. Los Mogabunos #245 Dpto. 306, Jesús María, Lima").

**Nombre completo de la madre**: Concatena `[Nombres] + [Apellidos]` (ej. "Susana Isabel Agurto Moran").

### BLOQUE 4: DATOS DEL PADRE
- **Apellidos**: En la fila "Apellidos" (ej. CÓRDOVA CRISANTO).
- **Nombres**: En la fila "Nombres" (ej. WALTER ANDRÉS).
- **DNI**: En la columna "DNI" (ej. 03664433).
- **Edad**: En la columna "Edad" (ej. 26).
- **Lugar de nacimiento**: En celdas "Departamento, Provincia, Distrito" (ej. "Piura, Piura, Piura").
- **Domicilio**: En la celda "Domicilio" (ej. "Jr. Los Mogabunos #245 Dpto. 306, Jesús María, Lima").

**Nombre completo del padre**: Concatena `[Nombres] + [Apellidos]` (ej. "Walter Andrés Córdova Crisanto").

### BLOQUE 5: DECLARANTE
- **Apellidos** y **Nombres** (ej. "ROSALES HUANCA GLADIS ESTER").
- **DNI** (ej. 08824442).
- **Relación con el nacido**: (ej. "MADRE", "PADRE").
- **Nombre completo**: Concatena `[Nombres] + [Apellidos]`.

### BLOQUE 6: REGISTRADOR(A)
- **Nombre completo** (Apellidos y Nombres).
- **Cargo**: (ej. "Registrador(a) Civil").

### BLOQUE 7: OBSERVACIONES / TESTIGOS
- **Observaciones**: texto libre al final (si hay).
- **Testigos**: Si hay nombres en la parte inferior, ponlos en el array. Si no, `[]`.

OJO: ESTE FORMATO SOLO ES PARA EL TIPO Tabular Moderno RENIEC.
"""

In [ ]:
PROMPT = PROMPT_BASE + PROMPT_F1 + PROMPT_F2

In [ ]:
PROMPT_TRANSCRIPCION = """
Eres un sistema experto en transcripción de documentos históricos y registrales del Perú.
Tu tarea es TRANSCRIBIR TODO el texto visible en la imagen, de forma literal y ordenada.
NO extraigas campos, NO resumas, NO interpretes. Solo transcribe y organiza. Deberás extraer
toda la información tal cual se presenta. Hay una region llamado "RENIEC ACTA" que es un codigo de barra.
Deberas extraer tambien eso en un apartado aparte como "codigo de barra RENIEC: " y el numero que figura 
"""

In [ ]:
PROMPT_NUEVO = """
Identifica que tipo de actas es si es de nacimiento, matrimonio o defunción. También el estado del 
documento en la imagen si es completa o deteriorada. De acuerdo al tipo de acta extrae información 
importante por ejemplo en el acta de nacimiento quizás la información importante es:

Acta de nacimiento.
Nombre: Juanito Alimaña
Padre: Hecto Conquistador Guevara
Madre: Juana Ruoerta

Para eso deberás LEER toda la imagen. Si aparece los apellidos del nacido ilegible deberás
completarlo con los apellido del padre y madre es decir el primer apellido del padre y de la madre
para completar el apellido del nacido. También deberá coincidir en nombre del nacido con los apellidos 
de los padres. Claro que para cada documento (defunción o matrinomio) deberás aplicar otros criterios.
También hay una sección llamada algo como RECTIFICACIÓN o algo asi deberás tomar en cuenta esa sección
para que agregues una parte de diga actualización y lo que diga esa parte. Por ejemplo en los de 
matrimonio quizás diga la fecha de divorcio.

"""

In [8]:
PROMPT_tag = """
Analiza esta acta del registro civil peruano y transcribe todo su texto.
Responde solo en TEXTO PLANO. No uses JSON, markdown ni bloques de código.
En etiquetas agrega este texto "MODELO: Qwen/Qwen2.5-VL-7B-Instruct"

PARTE 1 — ETIQUETAS
Escribe primero estas tres líneas, usando SOLO los valores permitidos:

TIPO_ACTA: NACIMIENTO | MATRIMONIO | DEFUNCION
ANOTACIONES_MARGINALES: VERDADERO | FALSO
ESTADO_IMAGEN: BUENO | REGULAR | MALO | CRITICO


Criterios:
- TIPO_ACTA: guíate por el título impreso (ej. "Partida de Nacimiento",
  "Registro Civil de Defunciones", "Matrimonios") y por el contenido.
- ANOTACIONES_MARGINALES: VERDADERO si hay texto añadido al margen o en un recuadro
  que modifica el acta. Suele contener palabras como "Divorcio", "Rectificación",
  "Nulidad", "Vista la sentencia", "En cumplimiento", "Juzgado", "Notario".
  Si no hay, FALSO.
- ESTADO_IMAGEN:
  BUENO: todo se lee con claridad.
  REGULAR: algunas palabras difíciles (tinta tenue, letra cursiva, sellos encima),
           pero se entiende el contenido.
  MALO: varias zonas ilegibles por manchas, tinta borrada o pliegues.
  CRITICO: faltan partes del documento o la mayor parte no se puede leer.

PARTE 2 — TRANSCRIPCIÓN
Después de las etiquetas, transcribe TODO el texto TAL CUAL APARECE.
- Primero toda la COLUMNA IZQUIERDA, de arriba hacia abajo.
- Luego toda la COLUMNA DERECHA, de arriba hacia abajo.
- Nunca mezcles líneas de una columna con la otra.
- Copia exactamente lo escrito. No corrijas ortografía ni conviertas números.
- No transcribas los puntos ni las rayas de relleno.
- Texto dudoso: [palabra?]. Ilegible: [ilegible].
- Firmas: [FIRMA]. Sellos: [SELLO: texto legible].

FORMATO DE SALIDA (omite las secciones que no existan):

TIPO_ACTA: ...
ANOTACIONES_MARGINALES: ...
ESTADO_IMAGEN: ...

COLUMNA IZQUIERDA
-----------------

COLUMNA DERECHA
---------------

FIRMAS Y SELLOS
---------------

ANOTACIONES MARGINALES
----------------------

OTROS TEXTOS
------------
"""

In [9]:
# Función principal que recibe una imagen PIL y devuelve un diccionario con los datos extraídos
def transcribir_acta(img) -> tuple[str, float]:

    imagen = preprocesar(img)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": imagen},
            {"type": "text",  "text": PROMPT_tag},
        ],
    }]

    texto_entrada = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    img_inputs, vid_inputs = process_vision_info(messages)

    inputs = qwen_processor(
        text=[texto_entrada],
        images=img_inputs,
        videos=vid_inputs,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    t_inicio = time.time()
    with torch.no_grad():
        out_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=2048,   # transcripción completa necesita más que 1024
            do_sample=False,
            repetition_penalty=1.05,
        )
    t_modelo = round(time.time() - t_inicio, 2)
    print(f"  [tiempo modelo] {t_modelo} segundos")

    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    texto = qwen_processor.batch_decode(
        trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    # Limpieza mínima: solo quitar cercos de código si el modelo los puso
    if texto.startswith("```"):
        texto = texto.strip("`").removeprefix("text").removeprefix("plaintext").strip()

    # Aviso si se cortó por límite de tokens
    if len(trimmed[0]) >= 2048:
        print("  [aviso] la salida llegó al límite de tokens, puede estar incompleta")

    torch.cuda.empty_cache()
    return texto, t_modelo

# Test modelo

In [10]:
CARPETA_IMAGENES = "/kaggle/input/datasets/andyjohan/actas-data/1-Grupo"
CARPETA_SALIDA   = "/kaggle/working/transcripciones"
EXTENSIONES = [".png", ".jpg", ".jpeg", ".tif", ".tiff"]

os.makedirs(CARPETA_SALIDA, exist_ok=True)


def probar_una_imagen(ruta_imagen):
    print(f"\n===== Probando: {ruta_imagen} =====")
    imagen = Image.open(ruta_imagen).convert("RGB")

    t0 = time.time()
    texto, t_modelo = transcribir_acta(imagen)
    t_total = time.time() - t0

    print(f"Tiempo modelo: {t_modelo} s | Total: {t_total:.2f} s")
    print("Texto extraído:\n")
    print(texto)  # string directo, se ven los saltos de línea
    print("====================================\n")

    # Guardar en .txt con el mismo nombre de la imagen
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0]
    with open(os.path.join(CARPETA_SALIDA, f"{nombre}.txt"), "w", encoding="utf-8") as f:
        f.write(texto)

    return texto


def probar_lote(carpeta):
    # Detecta extensiones sin importar mayúsculas/minúsculas
    archivos = sorted(
        os.path.join(carpeta, f) for f in os.listdir(carpeta)
        if os.path.splitext(f)[1].lower() in EXTENSIONES
    )

    if not archivos:
        print("No se encontraron imágenes en:", carpeta)
        return

    print(f"Se encontraron {len(archivos)} imágenes. Iniciando pruebas...\n")

    for ruta in archivos:
        try:
            probar_una_imagen(ruta)
        except Exception as e:
            # Que una imagen problemática no detenga todo el lote
            print(f"[ERROR] {ruta}: {e}")
            torch.cuda.empty_cache()

#probar_lote(CARPETA_IMAGENES)

In [ ]:
probar_una_imagen("/kaggle/input/datasets/andyjohan/actas-data/1-Grupo/1013061739.jpg")

# Otra forma

In [11]:
# ── 1. Encontrar imágenes ─────────────────────────────────
def buscar_imagenes(carpeta):
    archivos = []
    for raiz, _, nombres in os.walk(carpeta):
        for f in nombres:
            if os.path.splitext(f)[1].lower() in EXTENSIONES:
                archivos.append(os.path.join(raiz, f))
    return sorted(archivos)


# ── 2. Cargar una imagen ──────────────────────────────────
def cargar_imagen(ruta):
    return Image.open(ruta).convert("RGB")


# ── 3. Transcribir ────────────────────────────────────────
# transcribir_acta(img) -> (texto, t_modelo)   ← ya la tienes, no cambia


# ── 4. Calcular ruta de salida ────────────────────────────
def ruta_salida(ruta_imagen, carpeta_base, carpeta_salida):
    relativa = os.path.relpath(ruta_imagen, carpeta_base)
    return os.path.join(carpeta_salida, os.path.splitext(relativa)[0] + ".txt")


# ── 5. Guardar texto ──────────────────────────────────────
def guardar_texto(texto, ruta):
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(texto)


# ── 6. Mostrar resultado ──────────────────────────────────
def mostrar_resultado(ruta_imagen, texto, t_modelo):
    print(f"\n===== {ruta_imagen} =====")
    print(f"Tiempo modelo: {t_modelo} s\n")
    print(texto)
    print("=" * 40)

In [12]:
# Pipeline para UNA imagen
def procesar_imagen(ruta, carpeta_base, carpeta_salida, mostrar=True):
    img = cargar_imagen(ruta)
    texto, t_modelo = transcribir_acta(img)
    destino = ruta_salida(ruta, carpeta_base, carpeta_salida)
    guardar_texto(texto, destino)

    if mostrar:
        mostrar_resultado(ruta, texto, t_modelo)

    return texto, t_modelo


# Pipeline para TODA la carpeta
def procesar_lote(carpeta_base, carpeta_salida, mostrar=True):
    archivos = buscar_imagenes(carpeta_base)

    if not archivos:
        print("No se encontraron imágenes en:", carpeta_base)
        return

    print(f"Se encontraron {len(archivos)} imágenes.\n")

    for i, ruta in enumerate(archivos, 1):
        print(f"[{i}/{len(archivos)}] {os.path.relpath(ruta, carpeta_base)}")
        try:
            procesar_imagen(ruta, carpeta_base, carpeta_salida, mostrar)
        except Exception as e:
            print(f"  [ERROR] {e}")
            torch.cuda.empty_cache()

In [ ]:
#/kaggle/input/datasets/andyjohan/actas-data/0-Grupo/pagina1_foto1.jpeg
def procesar_imagen(ruta, carpeta_base, carpeta_salida, mostrar=True):

In [15]:
#CARPETA_IMAGENES = "/kaggle/input/datasets/andyjohan/actas-data/2-Grupo"
CARPETA_SALIDA   = "/kaggle/working/transcripcionese"
CARPETA_BASE = "/kaggle/input/datasets/andyjohan/actas-data/2-Grupo"
procesar_lote(CARPETA_BASE, CARPETA_SALIDA, mostrar=False)

Se encontraron 14 imágenes.

[1/14] 1025147536 DEF CASTILLA (6).jpg
  [tiempo modelo] 76.35 segundos
[2/14] 1025147536 DEF CASTILLA.jpg
  [tiempo modelo] 76.72 segundos
[3/14] 1025147537 DEF CASTILLA (7).jpg
  [tiempo modelo] 67.32 segundos
[4/14] 1025147537 DEF CASTILLA.jpg
  [tiempo modelo] 67.12 segundos
[5/14] 1025152700 DEF CASTILLA (3).jpg
  [tiempo modelo] 74.08 segundos
[6/14] 1025152700 DEF CASTILLA.jpg
  [tiempo modelo] 74.1 segundos
[7/14] 1025177090 MAT CASTILLA (5).jpg
  [tiempo modelo] 67.1 segundos
[8/14] 1025177090 MAT CASTILLA.jpg
  [tiempo modelo] 67.04 segundos
[9/14] 1025183819 MAT CASTILLA (4).jpg
  [tiempo modelo] 254.44 segundos
  [aviso] la salida llegó al límite de tokens, puede estar incompleta
[10/14] 1025183819 MAT CASTILLA.jpg
  [tiempo modelo] 254.79 segundos
  [aviso] la salida llegó al límite de tokens, puede estar incompleta
[11/14] 1025236331 MAT JAYANCA (2).jpg
  [tiempo modelo] 92.95 segundos
[12/14] 1025236331 MAT JAYANCA.jpg
  [tiempo modelo] 92.92

In [16]:
CARPETA_IMAGENES = "/kaggle/input/datasets/andyjohan/actas-data/1-Grupo/1013061739.jpg"
CARPETA_SALIDA   = "/kaggle/working/transcripcionese"
CARPETA_BASE = "/kaggle/input/datasets/andyjohan/actas-data"
procesar_imagen(CARPETA_IMAGENES, CARPETA_BASE,CARPETA_SALIDA, mostrar=True)

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (110897514 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  [tiempo modelo] 45.7 segundos

===== /kaggle/input/datasets/andyjohan/actas-data/1-Grupo/1013061739.jpg =====
Tiempo modelo: 45.7 s

MODELO: Qwen/Qwen2.5-VL-7B-Instruct

TIPO_ACTA: NACIMIENTO
ANOTACIONES_MARGINALES: FALSO
ESTADO_IMAGEN: REGULAR

COLUMNA IZQUIERDA
-----------------
Nombre: Aguilar Leandro
Fecha: 13 marzo 1922

COLUMNA DERECHA
---------------
Partida número: Sesenta
En Lima, a la Escritura y tres anotat de la tarde del día diez jueves de marzo de mil novecientos veintidós Don Gregorio Aguilar de cuarenta y nueve años casado término natural de Lima domiciliado en la calle de Homalgazque número trescientos tres manifestó un varón nacido el día trece de este mes a las siete de la noche en la calle de Homalgazque número trescientos doce llamado Leandro Aguilar hijo legítimo del declarante y de Doña Juana Guerara de Niemtz Priché cuyo natural de Lima Presentó como testigos a Don Luis Delgado electricista domiciliado en la calle de Junay número ciento cincuenta y seis y Don 

('MODELO: Qwen/Qwen2.5-VL-7B-Instruct\n\nTIPO_ACTA: NACIMIENTO\nANOTACIONES_MARGINALES: FALSO\nESTADO_IMAGEN: REGULAR\n\nCOLUMNA IZQUIERDA\n-----------------\nNombre: Aguilar Leandro\nFecha: 13 marzo 1922\n\nCOLUMNA DERECHA\n---------------\nPartida número: Sesenta\nEn Lima, a la Escritura y tres anotat de la tarde del día diez jueves de marzo de mil novecientos veintidós Don Gregorio Aguilar de cuarenta y nueve años casado término natural de Lima domiciliado en la calle de Homalgazque número trescientos tres manifestó un varón nacido el día trece de este mes a las siete de la noche en la calle de Homalgazque número trescientos doce llamado Leandro Aguilar hijo legítimo del declarante y de Doña Juana Guerara de Niemtz Priché cuyo natural de Lima Presentó como testigos a Don Luis Delgado electricista domiciliado en la calle de Junay número ciento cincuenta y seis y Don Carlos Samuel Barragán mecánico domiciliado en la calle de Comayagua número ciento treinta y tres ambos mayores de edad

In [ ]:
# Una sola imagen para probar
texto, t = procesar_imagen(
    f"{CARPETA_IMAGENES}/3-Grupo/1004226970 MAC BELLAVISTA.jpg",
    CARPETA_IMAGENES, CARPETA_SALIDA,
)

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/transcripcionese",   # nombre del zip (sin .zip)
    "zip",
    "/kaggle/working/transcripcionese",   # carpeta a comprimir
)

In [ ]:
import shutil
from IPython.display import FileLink, display

def descargar_resultados(carpeta_salida):
    nombre = os.path.basename(carpeta_salida.rstrip("/"))
    zip_path = shutil.make_archive(f"/kaggle/working/{nombre}", "zip", carpeta_salida)
    print(f"ZIP creado: {zip_path}")
    display(FileLink(os.path.relpath(zip_path, "/kaggle/working")))

In [ ]:
descargar_resultados(CARPETA_SALIDA)

# QWEN 3.0 VL

In [ ]:
# Isntalación de dependencias necesarias para ejecutar el proyecto
!pip install fastapi uvicorn python-multipar -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflaredt
!pip install bitsandbytes -q
!pip install "transformers>=4.49.0" -q
!pip install "qwen-vl-utils>=0.0.10" -q
#------------------------------------------
!pip install -U transformers accelerate
print("✓ Listo. Reinicia el kernel y ejecuta desde la siguiente celda.")

In [ ]:
# Librerias para el servidor FastAPI y el modelo Qwen2.5-VL.
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, HTMLResponse
import uvicorn, threading, io, subprocess, time, re
from PIL import Image
import json
from pathlib import Path

import torch
#from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

import cv2
import numpy as np
import json

import os
import glob

#----------------------------------
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

In [ ]:
DEVICE

In [ ]:
# Solo aplicar si vas a cuantizar
# Para modelo de 4 bits (Cuantizacion)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# Carga del modelo sin cuantizar
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # las T4 no soportan bfloat16
    device_map="auto",           # reparte entre las 2 T4 (~18 GB en fp16)
)
qwen_model.eval()

In [ ]:
# Cargar del modelo y procesador
qwen_processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
# Resolución: en Qwen3-VL cada token visual cubre 32x32 px (no 28x28)
TOKENS_MIN, TOKENS_MAX = 256, 4000
qwen_processor.image_processor.size = {
    "shortest_edge": TOKENS_MIN * 32 * 32,
    "longest_edge":  TOKENS_MAX * 32 * 32,
}
MAX_PIXELS = TOKENS_MAX * 32 * 32   # para tu preprocesar()

In [ ]:
# Info del vram utilizado
if torch.cuda.is_available():
    usado = torch.cuda.memory_allocated() / 1e9
    libre = torch.cuda.get_device_properties(0).total_memory / 1e9 - usado
    print(f"✓ Qwen2.5-VL-7B cargado | VRAM usada: {usado:.1f} GB | libre: {libre:.1f} GB")

In [ ]:
# Filtros usados

def upscale_lanczos_cv2(img, factor=2):
    h, w = img.shape[:2]
    return cv2.resize(img, (w * factor, h * factor), interpolation=cv2.INTER_LANCZOS4)

def _clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    return cv2.cvtColor(cv2.merge([clahe.apply(L), A, B]), cv2.COLOR_LAB2BGR)

def _sharp(img):
    k = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
    return cv2.addWeighted(img, 0.4, cv2.filter2D(img, -1, k), 0.6, 0)

# Funcion que hace el procesamiento de imagenes
def preprocesar2(img):
    
    img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    img = upscale_lanczos_cv2(img)
    img = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    img = _clahe(img)
    img = _sharp(img)

    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))


MAX_PIXELS = 4000 * 28 * 28   # el mismo que pusiste en el processor

def preprocesar(img, filtros_baja_res=True):
    w, h = img.size
    pixeles = w * h

    # Imagen de buena calidad: no tocar
    if pixeles >= MAX_PIXELS:
        return img

    # Imagen de baja resolución: aplicar filtros
    if filtros_baja_res:
        arr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        # Escalar solo lo necesario para llegar a MAX_PIXELS (no un ×2 fijo)
        factor = (MAX_PIXELS / pixeles) ** 0.5
        arr = cv2.resize(arr, (int(w * factor), int(h * factor)),
                         interpolation=cv2.INTER_LANCZOS4)
        arr = _clahe(arr)
        arr = _sharp(arr)
        return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))

    return img

In [ ]:
PROMPT_BASE = """
Eres un sistema experto en transcripción de documentos históricos y registrales del Perú.
Tu tarea es TRANSCRIBIR TODO el texto visible en la imagen, de forma literal y ordenada.
NO extraigas campos, NO resumas, NO interpretes. Solo transcribe y organiza.

Analiza la imagen y determina primero a cuál de los dos formatos corresponde:
- TIPO A: Manuscrito Antiguo (2 columnas: lateral izquierdo + declaración central)
- TIPO B: Tabular Moderno RENIEC (bloques con casillas)

Aplica ÚNICAMENTE las reglas del tipo que corresponda. Las reglas de un tipo no aplican al otro.
Debes analizar la imagen adjunta y extraer la información solicitada.

INSTRUCCIONES SEGÚN EL TIPO DE ACTA:
"""

PROMPT_F1 = """
1. SI EL ACTA TIENE 2 columnas - Declaracion lateral izquierdo / Declaración central derecha:
   A. INFORMACIÓN DE LA SECCION IZQUIERDA:
    Extraer toda información de la imagen. 

   B. SECCION DERECHA
	En esta sección deberas extraer toda la información tal cual se presenta. 
    También encontrar un código de barras. Deberas poner esa información en una 
    sección llamada "Datos barras" y colocar los números que figuran.
    """

PROMPT_F2 = """
SI EL ACTA ES TABULAR RENIEC ES DECIR la imagen es un acta con formato de tabla y bloques horizontales. 
Las etiquetas de cada campo están **DEBAJO** de las casillas punteadas, no encima.

**REGLA DE ORO (AISLAMIENTO TOTAL):**
Cada bloque es independiente. **PROHIBIDO** usar información de un bloque para rellenar otro. 
Si un campo no está en su bloque, déjalo como `null`.

---

### BLOQUE 1: OFICINA REGISTRAL (Parte superior izquierda)
- **Fecha de registro**: día/mes/año (ej. 21/12/2000). Está en la primera fila.
- **Distrito**: (ej. JESÚS MARÍA). 
- **Provincia**: (ej. LIMA).
- **Departamento**: (ej. LIMA).
- **Oficina registral**: Distrito, Provincia, Departamento (ej. "Jesús María, Lima, Lima").

### BLOQUE 2: DATOS DEL NACIDO
- **Apellidos**: En la fila "Apellidos" (ej. CÓRDOVA AGURTO).
- **Nombres**: En la fila "Nombres" (ej. ANDREA SUSANA).
- **Sexo**: En la fila "Sexo" (ej. 2 FEMENINO).
- **Fecha de nacimiento**: En las celdas de día/mes/año (ej. 21/12/2000).
- **Hora de nacimiento**: En la celda "Hora" (ej. 11:25 AM).
- **Lugar de nacimiento**: En las celdas "Departamento", "Provincia", "Distrito" (ej. "Hospital Rebagliati, Lima, Lima").

**Nombre completo del nacido**: Concatena `[Nombres] + [Apellidos]` (ej. "Andrea Susana Córdova Agurto").

### BLOQUE 3: DATOS DE LA MADRE
- **Apellidos**: En la fila "Apellidos" (ej. AGURTO MORAN).
- **Nombres**: En la fila "Nombres" (ej. SUSANA ISABEL).
- **DNI**: En la columna "DNI" (ej. 03690243).
- **Edad**: En la columna "Edad" (ej. 22).
- **Lugar de nacimiento**: En celdas "Departamento, Provincia, Distrito" (ej. "Piura, Piura, Piura").
- **Domicilio**: En la celda "Domicilio" (ej. "Jr. Los Mogabunos #245 Dpto. 306, Jesús María, Lima").

**Nombre completo de la madre**: Concatena `[Nombres] + [Apellidos]` (ej. "Susana Isabel Agurto Moran").

### BLOQUE 4: DATOS DEL PADRE
- **Apellidos**: En la fila "Apellidos" (ej. CÓRDOVA CRISANTO).
- **Nombres**: En la fila "Nombres" (ej. WALTER ANDRÉS).
- **DNI**: En la columna "DNI" (ej. 03664433).
- **Edad**: En la columna "Edad" (ej. 26).
- **Lugar de nacimiento**: En celdas "Departamento, Provincia, Distrito" (ej. "Piura, Piura, Piura").
- **Domicilio**: En la celda "Domicilio" (ej. "Jr. Los Mogabunos #245 Dpto. 306, Jesús María, Lima").

**Nombre completo del padre**: Concatena `[Nombres] + [Apellidos]` (ej. "Walter Andrés Córdova Crisanto").

### BLOQUE 5: DECLARANTE
- **Apellidos** y **Nombres** (ej. "ROSALES HUANCA GLADIS ESTER").
- **DNI** (ej. 08824442).
- **Relación con el nacido**: (ej. "MADRE", "PADRE").
- **Nombre completo**: Concatena `[Nombres] + [Apellidos]`.

### BLOQUE 6: REGISTRADOR(A)
- **Nombre completo** (Apellidos y Nombres).
- **Cargo**: (ej. "Registrador(a) Civil").

### BLOQUE 7: OBSERVACIONES / TESTIGOS
- **Observaciones**: texto libre al final (si hay).
- **Testigos**: Si hay nombres en la parte inferior, ponlos en el array. Si no, `[]`.

OJO: ESTE FORMATO SOLO ES PARA EL TIPO Tabular Moderno RENIEC.
"""

PROMPT_S = """Encontraras imágenes que no coincida con ninguno de los formato. Estos imágenes tienen 
la siguiente característica: son hojas sin ningún formato solo escrito manualmente describiendo el 
acontecimiento (nacimiento, casamiento, fallecimiento entre otros) Esas imágenes deberás transcribir 
tal cual aparecen."""

PROMPT = PROMPT_BASE + PROMPT_F1 + PROMPT_F2 + PROMPT_S

In [ ]:
INICIO = "COLUMNA IZQUIERDA\n-----------------\n"

def transcribir_acta(img) -> tuple[str, float]:
    imagen = preprocesar(img)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": imagen},
            {"type": "text",  "text": PROMPT},
        ],
    }]

    # Plantilla como texto + pre-relleno del formato
    texto_entrada = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    ) + INICIO

    inputs = qwen_processor(
        text=[texto_entrada],
        images=[imagen],
        return_tensors="pt",
    ).to(qwen_model.device)

    t_inicio = time.time()
    with torch.no_grad():
        out_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            repetition_penalty=1.05,
        )
    t_modelo = round(time.time() - t_inicio, 2)
    print(f"  [tiempo modelo] {t_modelo} segundos")

    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    texto = INICIO + qwen_processor.batch_decode(
        trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    if len(trimmed[0]) >= 2048:
        print("  [aviso] la salida llegó al límite de tokens, puede estar incompleta")

    torch.cuda.empty_cache()
    return texto, t_modelo

In [ ]:
CARPETA_IMAGENES = "/kaggle/input/datasets/andyjohan/actas-data/2-Grupo"
CARPETA_SALIDA   = "/kaggle/working/transcripciones"
EXTENSIONES = [".png", ".jpg", ".jpeg", ".tif", ".tiff"]

os.makedirs(CARPETA_SALIDA, exist_ok=True)


def probar_una_imagen(ruta_imagen):
    print(f"\n===== Probando: {ruta_imagen} =====")
    imagen = Image.open(ruta_imagen).convert("RGB")

    t0 = time.time()
    texto, t_modelo = transcribir_acta(imagen)
    t_total = time.time() - t0

    print(f"Tiempo modelo: {t_modelo} s | Total: {t_total:.2f} s")
    print("Texto extraído:\n")
    print(texto)  # string directo, se ven los saltos de línea
    print("====================================\n")

    # Guardar en .txt con el mismo nombre de la imagen
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0]
    with open(os.path.join(CARPETA_SALIDA, f"{nombre}.txt"), "w", encoding="utf-8") as f:
        f.write(texto)

    return texto


def probar_lote(carpeta):
    # Detecta extensiones sin importar mayúsculas/minúsculas
    archivos = sorted(
        os.path.join(carpeta, f) for f in os.listdir(carpeta)
        if os.path.splitext(f)[1].lower() in EXTENSIONES
    )

    if not archivos:
        print("No se encontraron imágenes en:", carpeta)
        return

    print(f"Se encontraron {len(archivos)} imágenes. Iniciando pruebas...\n")

    for ruta in archivos:
        try:
            probar_una_imagen(ruta)
        except Exception as e:
            # Que una imagen problemática no detenga todo el lote
            print(f"[ERROR] {ruta}: {e}")
            torch.cuda.empty_cache()

#probar_lote(CARPETA_IMAGENES)

In [ ]:
#probar_una_imagen("/kaggle/input/datasets/andyjohan/actas-data/2-Grupo/1025147536 DEF CASTILLA (6).jpg")

In [ ]:
# ── 1. Encontrar imágenes ─────────────────────────────────
def buscar_imagenes(carpeta):
    archivos = []
    for raiz, _, nombres in os.walk(carpeta):
        for f in nombres:
            if os.path.splitext(f)[1].lower() in EXTENSIONES:
                archivos.append(os.path.join(raiz, f))
    return sorted(archivos)


# ── 2. Cargar una imagen ──────────────────────────────────
def cargar_imagen(ruta):
    return Image.open(ruta).convert("RGB")


# ── 3. Transcribir ────────────────────────────────────────
# transcribir_acta(img) -> (texto, t_modelo)   ← ya la tienes, no cambia


# ── 4. Calcular ruta de salida ────────────────────────────
def ruta_salida(ruta_imagen, carpeta_base, carpeta_salida):
    relativa = os.path.relpath(ruta_imagen, carpeta_base)
    return os.path.join(carpeta_salida, os.path.splitext(relativa)[0] + ".txt")


# ── 5. Guardar texto ──────────────────────────────────────
def guardar_texto(texto, ruta):
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(texto)


# ── 6. Mostrar resultado ──────────────────────────────────
def mostrar_resultado(ruta_imagen, texto, t_modelo):
    print(f"\n===== {ruta_imagen} =====")
    print(f"Tiempo modelo: {t_modelo} s\n")
    print(texto)
    print("=" * 40)

In [ ]:
# Pipeline para UNA imagen
def procesar_imagen(ruta, carpeta_base, carpeta_salida, mostrar=True):
    img = cargar_imagen(ruta)
    texto, t_modelo = transcribir_acta(img)
    destino = ruta_salida(ruta, carpeta_base, carpeta_salida)
    guardar_texto(texto, destino)

    if mostrar:
        mostrar_resultado(ruta, texto, t_modelo)

    return texto, t_modelo


# Pipeline para TODA la carpeta
def procesar_lote(carpeta_base, carpeta_salida, mostrar=True):
    archivos = buscar_imagenes(carpeta_base)

    if not archivos:
        print("No se encontraron imágenes en:", carpeta_base)
        return

    print(f"Se encontraron {len(archivos)} imágenes.\n")

    for i, ruta in enumerate(archivos, 1):
        print(f"[{i}/{len(archivos)}] {os.path.relpath(ruta, carpeta_base)}")
        try:
            procesar_imagen(ruta, carpeta_base, carpeta_salida, mostrar)
        except Exception as e:
            print(f"  [ERROR] {e}")
            torch.cuda.empty_cache()

In [ ]:
ruta = "/kaggle/input/datasets/andyjohan/actas-data/1-Grupo/1013061739.jpg"

In [ ]:
img = Image.open(ruta).convert("RGB")
inputs = qwen_processor(text=["<|vision_start|><|image_pad|><|vision_end|>"],
                        images=[img], return_tensors="pt")
t, h, w = inputs["image_grid_thw"][0].tolist()
print(f"Grid: {h}x{w} parches → {h * w // 4} tokens visuales")

In [ ]:
actas = ["/kaggle/input/datasets/andyjohan/actas-data/0-Grupo/pagina2_foto1.jpeg",
"/kaggle/input/datasets/andyjohan/actas-data/0-Grupo/pagina4_foto1.jpeg",
"/kaggle/input/datasets/andyjohan/actas-data/3-Grupo/1007233857 NAC IQUITOS.jpg"]


CARPETA_SALIDA   = "/kaggle/working/transcripcionese"
CARPETA_BASE = "/kaggle/input/datasets/andyjohan/actas-data"

for i in actas:
    print(i)
    CARPETA_IMAGENES = i
    procesar_imagen(CARPETA_IMAGENES, CARPETA_BASE,CARPETA_SALIDA, mostrar=True)

In [ ]:
CARPETA_IMAGENES = "/kaggle/input/datasets/andyjohan/actas-data/3-Grupo/1006914341 NAC PIURA.jpg"
CARPETA_SALIDA   = "/kaggle/working/transcripcionese"
CARPETA_BASE = "/kaggle/input/datasets/andyjohan/actas-data"
procesar_imagen(CARPETA_IMAGENES, CARPETA_BASE,CARPETA_SALIDA, mostrar=True)

In [ ]:
CARPETA_IMAGENES = "/kaggle/input/datasets/andyjohan/actas-data/1-Grupo"
CARPETA_SALIDA   = "/kaggle/working/transcripciones"

# Todo el lote, sin imprimir cada texto (más limpio con muchas actas)
procesar_lote(CARPETA_IMAGENES, CARPETA_SALIDA)#, mostrar=False)


# SERVIDOR

In [ ]:
# Servidor 
import subprocess, time, re, threading, io, base64
subprocess.run(["fuser", "-k", "7860/tcp"], capture_output=True)
time.sleep(2)

from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn, asyncio

CARPETA_JSON = Path("/kaggle/working/resultados_actas")
CARPETA_JSON.mkdir(exist_ok=True)

app = FastAPI()
app.add_middleware(CORSMiddleware,
                   allow_origins=["*"],
                   allow_methods=["*"],
                   allow_headers=["*"])

def img_a_b64(pil_img, max_w=900):
    w,h = pil_img.size
    if w > max_w:
        pil_img = pil_img.resize((max_w, int(h*max_w/w)), Image.LANCZOS)
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

@app.get("/ping")
def ping(): return {"status": "ok", "modelo": "Qwen2.5-VL-7B"}

@app.post("/transcribir")
async def transcribir(file: UploadFile = File(...)):
    
    imagen = Image.open(io.BytesIO(await file.read())).convert("RGB")

    # ── INICIO cronómetro total ────────────────────────────────
    t_total_inicio = time.time()
    # ──────────────────────────────────────────────────────────
    
    campos, t_modelo = transcribir_acta(imagen)

    # ── FIN cronómetro total ──────────────────────────────────
    t_total = round(time.time() - t_total_inicio, 2)
    print(f"  [tiempo total]  {t_total} segundos")
    # ──────────────────────────────────────────────────────────   

        # ── Guardar JSON en disco ──────────────────────────────────
    # Nombre del archivo: timestamp para que no se sobreescriban
    from datetime import datetime
    timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
    nombre_json = f"acta_{timestamp}.json"
    ruta_json   = CARPETA_JSON / nombre_json

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(campos, f, ensure_ascii=False, indent=2)

    print(f"[guardado] {ruta_json}")
    # ──────────────────────────────────────────────────────────

    # Devolver también la imagen preprocesada para mostrar en el front
    img_proc = preprocesar(imagen)

    return JSONResponse({
        "campos":  campos,
        "imagen":  img_a_b64(img_proc),
        "archivo_guardado": nombre_json,# imagen preprocesada sin boxes
        "tiempo_modelo_seg": t_modelo,   # ← tiempo solo del modelo
        "tiempo_total_seg":  t_total,    # ← tiempo total incluyendo todo
    })



def _run():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    cfg = uvicorn.Config(app, host="0.0.0.0", port=7860,
                         log_level="error", loop="none")
    loop.run_until_complete(uvicorn.Server(cfg).serve())

threading.Thread(target=_run, daemon=True).start()
time.sleep(3)
print("✓ Servidor listo")

proc = subprocess.Popen(
    ["./cloudflared","tunnel","--url","http://localhost:7860"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
)
for _ in range(30):
    linea = proc.stderr.readline().decode("utf-8", errors="ignore")
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", linea)
    if m:
        print(f"\n✅ URL    : {m.group()}")
        print(f"✅ Ping   : {m.group()}/ping")
        break
        
    time.sleep(1)